# S04: mini project, a verifiable policy service

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/notebooks/03_mini_knowledge_service.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

The notebook anchors the 90-minute build block inside a full three-hour mini project. Produce evidence that an answer is correct, authorised and supported. A source ID alone is inadequate.

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–20 min | Run the service and inspect evidence.** Ask the standard payment question, a restricted discount question and an unknown question. In rehearsal you get actual retrieved passages. Generated answers and semantic acceptance require live calls.

In [ ]:
from engine.evaluate import semantic_review
ix=Index();ix.build('v1',read_data('policies.json'));db=Purchases();app=Platform(ix,db,client)
for question in ['What are standard supplier payment terms?','What is the negotiated discount schedule?','What is the annual holiday allowance?']:
    print(question, '\n', json.dumps(app.ask(BUYER,'policy',question),indent=2))

**20–40 min | Demonstrate a misleading citation.** The answer below cites a real, exact quote but makes the opposite business claim. Observe the structural validator pass. The validator checks provenance syntax, not semantic entailment. Reviewers must examine the supported meaning.

In [ ]:
hits=ix.search('standard supplier payment terms',BUYER)
source=next(h for h in hits if h['id']=='A-TERMS')
misleading={'status':'answered','answer':'Standard suppliers are paid in 90 days.',
            'evidence':[{'id':source['id'],'quote':source['text']}]}
assert evidence_errors(misleading,hits)==[]
print('Structural errors:',evidence_errors(misleading,hits))
print('Human review: INCORRECT. The quoted source says 45 days, not 90.')

**40–65 min | Build the acceptance evidence.** Use nine cases including a partially answerable question. For every live answer, mark correct, supported and complete separately. Preserve failures; do not edit actual outputs to match labels. A refusal may be appropriate but must not invent a reason. `None` means unreviewed.

In [ ]:
review=semantic_review(app)
Path('outputs').mkdir(exist_ok=True)
Path('outputs/mini-project-review.json').write_text(json.dumps(review,indent=2))
print('Saved',len(review),'cases; human review remains pending.')
print(json.dumps(review[-1],indent=2))

**65–90 min | Implement one improvement.** Choose source-aware chunking, clearer missing-fact responses, or a permission-revocation design. Add two new tests before changing the code/prompt. Show the failed case, the change and the new result. Do not merely rerun the supplied cells.

**Team deliverable:** service demonstration; one architecture diagram; two new cases; nine reviewed answers if live; one limitation and one improvement. During the five 8-minute demos, use 3 minutes for behavior, 3 for failure evidence and 2 for questions. Assignment 1 turns this into an individual ADR. Close resources when finished.

In [ ]:
ix.close();db.close()